In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup

In [2]:
!pip install -q faiss-cpu

import torch, pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

DATA = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ["A", "B", "C", "D", "E"]
DEV = 0 if torch.cuda.is_available() else -1

train = pd.read_csv(f"{DATA}/train.csv")

kb = [str(row[row["answer"]]) for _, row in train.iterrows()]

model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = model.encode(kb, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("KB + FAISS index ready:", index.ntotal, "docs")

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEV)

def retrieve(prompt, k):
    """top-k KB indices for a prompt (bi-encoder retrieval)."""
    q = model.encode([str(prompt)]).astype("float32")
    D, I = index.search(q, k)
    return I[0]

def correct_prob(text, labels, correct_text):
    """zero-shot: probability assigned to the correct option."""
    res = zs(text, candidate_labels=labels)
    return dict(zip(res["labels"], res["scores"]))[correct_text]

row_150   = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150[o]) for o in OPTIONS]
ans_150    = str(row_150[row_150["answer"]])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.0 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


**Zero-shot classifier for Q1, Q2, Q6**
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 

row_150 = train.iloc[150] 

prompt_150 = str(row_150['prompt']) 

labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 

ans_150 = str(row_150[row_150['answer']])

**Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)**

In [3]:
Q1 = round(correct_prob(prompt_150, labels_150, ans_150), 3)
print("Q1:", Q1)

Q1: 0.384


**Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?**

In [4]:
idx10 = retrieve(prompt_150, 10)
Q2 = int(np.where(idx10 == 150)[0][0]) + 1 if 150 in idx10 else "not in top 10"
print("Q2 (FAISS rank of true doc):", Q2)

Q2 (FAISS rank of true doc): 10


**Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)**

In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 

A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

[https://huggingface.co/cross-encoder](http://)

**Code to use a Cross-Encoder**

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks

pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs

ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

**Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?**

In [5]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
docs_10 = [kb[i] for i in idx10]
ce_scores = cross_encoder.predict([[prompt_150, d] for d in docs_10])
reranked = idx10[np.argsort(ce_scores)[::-1]]          # sort by CE score desc
Q3 = int(np.where(reranked == 150)[0][0]) + 1 if 150 in reranked else "not found"
print("Q3 (cross-encoder rank of true doc):", Q3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3 (cross-encoder rank of true doc): 1


**Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?**

In [6]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
prompt_42 = str(train.iloc[42]["prompt"])
docs5_42  = [kb[i] for i in retrieve(prompt_42, 5)]
rag_42    = f"Context: {' '.join(docs5_42)} Question: {prompt_42}"
Q4 = len(bert_tok(rag_42, truncation=False)["input_ids"])
print("Q4 (total tokens):", Q4)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 (total tokens): 216


**Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).**

In [7]:
rag5 = f"Context: {kb[150]} Question: {prompt_150}"
Q5 = round(correct_prob(rag5, labels_150, ans_150), 3)
print("Q5 (prob with correct context):", Q5)

Q5 (prob with correct context): 0.989


**Concept: The Danger of Bad Retrieval (Adversarial RAG)**

The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.


**Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).**

In [8]:
rag6 = f"Context: {kb[999]} Question: {prompt_150}"
Q6 = round(correct_prob(rag6, labels_150, ans_150), 3)
print("Q6 (prob with WRONG context):", Q6)

Q6 (prob with WRONG context): 0.529


*In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.*

**Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).**

In [9]:
hits = 0
for i in range(100):
    r = train.iloc[i]
    correct = str(r[r["answer"]])
    docs = [kb[j] for j in retrieve(str(r["prompt"]), 5)]
    if any(correct in d for d in docs):
        hits += 1
Q7 = round(100 * hits / 100, 1)
print("Q7 (Hit Rate %):", Q7)

Q7 (Hit Rate %): 73.0


**Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.**

For each row, your pipeline must do the following in order:

**Retrieve:** Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

**Rerank:** Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

**Augment:** Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

**Score:** Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

**What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).**



In [10]:
def map3_row(ranked_letters, truth):
    for i, l in enumerate(ranked_letters[:3]):
        if l == truth:
            return 1.0 / (i + 1)
    return 0.0

maps = []
for i in range(20):
    r = train.iloc[i]
    prompt = str(r["prompt"])
    labels = [str(r[o]) for o in OPTIONS]
    lab2letter = {str(r[o]): o for o in OPTIONS}

    # 1. Retrieve top-5
    docs5 = [kb[j] for j in retrieve(prompt, 5)]
    # 2. Rerank -> best doc
    ce = cross_encoder.predict([[prompt, d] for d in docs5])
    best_doc = docs5[int(np.argmax(ce))]
    # 3. Augment
    rag = f"Context: {best_doc} Question: {prompt}"
    # 4. Predict (zero-shot) -> ranked letters
    res = zs(rag, candidate_labels=labels)
    ranked_letters = [lab2letter[lab] for lab in res["labels"]]
    # 5. MAP@3
    maps.append(map3_row(ranked_letters, r["answer"]))

Q8 = round(float(np.mean(maps)), 3)
print("Q8 (RAG pipeline MAP@3):", Q8)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q8 (RAG pipeline MAP@3): 0.975


In [11]:
print("MILESTONE 3 - ANSWERS")
for k, v in dict(Q1=Q1, Q2=Q2, Q3=Q3, Q4=Q4, Q5=Q5, Q6=Q6, Q7=Q7, Q8=Q8).items():
    print(f"{k}: {v}")

MILESTONE 3 - ANSWERS
Q1: 0.384
Q2: 10
Q3: 1
Q4: 216
Q5: 0.989
Q6: 0.529
Q7: 73.0
Q8: 0.975
